# Fine-tuning YOLO11x para Basketball Analytics

Entrena un detector especializado en básquet (jugadores + pelota + aro + árbitro)
sobre un dataset de **Roboflow Universe**, usando **GPU de Colab**.

Resuelve los dos problemas abiertos del POC:
1. YOLO genérico no detecta la pelota en el arco del tiro → 0 canastas.
2. Asignación de equipos floja por detección de jugadores poco fiable.

**Workflow:** Runtime → Change runtime type → **GPU (T4/A100)** → ejecutar todo.

Al terminar, descargás `best.pt` y lo usás en el POC así:
```bash
python basketball_poc_v3.py --source video.mp4 --weights best.pt \
    --player-class <ID_player> --ball-class <ID_ball>
```
Los `class_id` correctos los imprime la celda 5 (data.yaml).

## 1. Verificar GPU

In [ ]:
!nvidia-smi
import torch
print('CUDA disponible:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NINGUNA — cambiá el runtime a GPU')

## 2. Instalar dependencias

In [ ]:
!pip install -q ultralytics roboflow supervision
import ultralytics; ultralytics.checks()

## 3. Descargar dataset de básquet (Roboflow)

Necesitás una **API key gratuita** de Roboflow: https://app.roboflow.com → Settings → API.

Pegala abajo. El ejemplo usa un dataset público de básquet de Roboflow Universe
(jugadores + pelota + aro). Podés cambiar `workspace/project/version` por cualquier
otro dataset de básquet de Universe que prefieras.

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = 'PEGA_TU_API_KEY_AQUI'  # <-- reemplazar

# Dataset de ejemplo (cambiable). Buscá más en https://universe.roboflow.com (query: basketball)
WORKSPACE = 'roboflow-jvuqo'
PROJECT   = 'basketball-players-fy4c2'
VERSION   = 14

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download('yolov11')
print('Dataset en:', dataset.location)

## 4. Inspeccionar el data.yaml — anotá los class_id

**IMPORTANTE:** el orden de `names` define los `class_id` que después pasás al POC
con `--player-class` y `--ball-class`.

In [ ]:
import yaml, os
data_yaml = os.path.join(dataset.location, 'data.yaml')
with open(data_yaml) as f:
    cfg = yaml.safe_load(f)
print('Clases del dataset (índice → nombre):')
names = cfg['names'] if isinstance(cfg['names'], list) else [cfg['names'][i] for i in sorted(cfg['names'])]
for i, n in enumerate(names):
    print(f'  {i}: {n}')
print('\n>>> Usá estos índices en --player-class y --ball-class al correr el POC <<<')

## 5. Entrenar YOLO11x

`yolo11x` = máxima precisión. En T4 conviene `imgsz=640`; en A100 podés subir a 1280
para detectar mejor la pelota (objeto chico). Ajustá `epochs` según el tiempo disponible.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11x.pt')  # se descarga el checkpoint COCO como punto de partida

results = model.train(
    data=data_yaml,
    epochs=100,
    imgsz=640,          # subí a 1280 en A100 para mejor recall de la pelota
    batch=16,           # bajá a 8 si hay OOM en T4
    patience=20,        # early stopping
    device=0,
    project='basketball_yolo',
    name='yolo11x_ft',
    plots=True,
)

## 6. Validar y ver métricas

In [ ]:
metrics = model.val()
print('mAP50-95:', metrics.box.map)
print('mAP50   :', metrics.box.map50)
print('Por clase (mAP50):')
for i, n in enumerate(names):
    try:
        print(f'  {n:20s}: {metrics.box.maps[i]:.3f}')
    except Exception:
        pass

In [ ]:
# Ver curvas de entrenamiento
from IPython.display import Image, display
import glob
run_dir = 'basketball_yolo/yolo11x_ft'
for img in ['results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg']:
    p = os.path.join(run_dir, img)
    if os.path.exists(p):
        display(Image(p, width=700))

## 7. Descargar los pesos entrenados

In [ ]:
best = os.path.join(run_dir, 'weights', 'best.pt')
print('Pesos:', best, '|', round(os.path.getsize(best)/1e6, 1), 'MB')

# Opción A: descarga directa al navegador
from google.colab import files
files.download(best)

# Opción B (recomendada para no perderlos): copiar a Google Drive
# from google.colab import drive; drive.mount('/content/drive')
# !cp {best} /content/drive/MyDrive/basketball_best.pt

## 8. Prueba rápida de inferencia (opcional)

Subí un frame o clip corto de tu video para ver cómo detecta el modelo nuevo.

In [ ]:
from google.colab import files as upfiles
up = upfiles.upload()  # subí una imagen .jpg/.png
for fname in up:
    r = model(fname, conf=0.25)[0]
    r.save(filename='pred_'+fname)
    display(Image('pred_'+fname, width=700))